# 챌린지: 데이터 과학에 관한 텍스트 분석하기

이 예제에서는 전통적인 데이터 과학 프로세스의 모든 단계를 포함하는 간단한 연습을 해보겠습니다. 코드를 직접 작성할 필요는 없으며, 아래 셀을 클릭하여 실행하고 결과를 관찰하기만 하면 됩니다. 도전 과제로서 이 코드를 다양한 데이터로 시도해보는 것을 권장합니다.

## 목표

이번 수업에서는 데이터 과학과 관련된 여러 개념을 논의해 왔습니다. <strong>텍스트 마이닝</strong>을 통해 더 관련된 개념들을 발견해 봅시다. 데이터 과학에 관한 텍스트를 시작으로, 키워드를 추출하고 그 결과를 시각화하는 것을 시도할 것입니다.

텍스트로는 위키피디아의 데이터 과학 페이지를 사용하겠습니다:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## 1단계: 데이터 가져오기

모든 데이터 과학 프로세스의 첫 번째 단계는 데이터를 가져오는 것입니다. 이를 위해 `requests` 라이브러리를 사용할 것입니다:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## 2단계: 데이터 변환

다음 단계는 데이터를 처리하기 적합한 형태로 변환하는 것입니다. 이번 경우에는 페이지에서 HTML 소스 코드를 다운로드했으므로, 이를 일반 텍스트로 변환해야 합니다.

이 작업을 수행하는 방법은 여러 가지가 있습니다. 여기서는 HTML 파싱을 위한 인기 있는 파이썬 라이브러리인 [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/)을 사용할 것입니다. BeautifulSoup을 사용하면 특정 HTML 요소를 타겟팅할 수 있어, 위키피디아의 주요 기사 내용에 집중하고 일부 탐색 메뉴, 사이드바, 푸터 및 기타 불필요한 내용을 줄일 수 있습니다(일부 고정 텍스트는 여전히 남을 수 있지만).


먼저, HTML 파싱을 위해 BeautifulSoup 라이브러리를 설치해야 합니다:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## 3단계: 인사이트 얻기

가장 중요한 단계는 데이터를 인사이트를 도출할 수 있는 형태로 변환하는 것입니다. 우리 경우에는 텍스트에서 키워드를 추출하고, 어느 키워드가 더 의미 있는지 확인하고자 합니다.

키워드 추출을 위해 [RAKE](https://github.com/aneesha/RAKE)라는 파이썬 라이브러리를 사용할 것입니다. 먼저, 이 라이브러리가 없을 경우 설치해 보겠습니다: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

주요 기능은 `Rake` 객체에서 사용할 수 있으며, 일부 매개변수를 사용하여 사용자 정의할 수 있습니다. 우리 경우에는 키워드의 최소 길이를 5자로, 문서 내 키워드의 최소 빈도를 3으로, 키워드 내 최대 단어 수를 2로 설정할 것입니다. 다른 값을 자유롭게 조정해보고 결과를 관찰해보세요.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


관련도와 함께 용어 목록을 얻었습니다. 보시다시피, 기계 학습과 빅 데이터와 같은 가장 관련성 높은 분야들이 목록 상위에 있습니다.

## 4단계: 결과 시각화

사람들은 데이터를 시각적 형태로 가장 잘 이해할 수 있습니다. 따라서 통찰을 얻기 위해 데이터를 시각화하는 것이 종종 의미가 있습니다. 우리는 Python의 `matplotlib` 라이브러리를 사용하여 키워드와 관련성의 간단한 분포를 시각화할 수 있습니다:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

하지만 단어 빈도를 시각화하는 훨씬 더 좋은 방법이 있습니다 - 바로 **워드 클라우드(Word Cloud)** 입니다. 키워드 목록에서 워드 클라우드를 그리기 위해 다른 라이브러리를 설치해야 합니다.


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` 객체는 원본 텍스트 또는 단어와 해당 빈도의 사전 계산된 목록을 받아 이미지로 반환하며, 이 이미지는 `matplotlib`을 사용하여 표시할 수 있습니다:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

원본 텍스트를 `WordCloud`에 전달할 수도 있습니다 - 비슷한 결과를 얻을 수 있는지 확인해봅시다:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

지금 보시면 단어 구름이 더 인상적으로 보이지만, 동시에 많은 노이즈(예: `Retrieved on`과 같은 관련 없는 단어)도 포함되어 있습니다. 또한 <em>데이터 과학자</em>나 <em>컴퓨터 과학</em>과 같이 두 단어로 이루어진 키워드가 적게 나타납니다. 이는 RAKE 알고리즘이 텍스트에서 좋은 키워드를 선택하는 데 훨씬 더 뛰어나기 때문입니다. 이 예시는 데이터 전처리와 정리의 중요성을 보여줍니다. 마지막에 명확한 결과가 있어야 더 좋은 결정을 내릴 수 있기 때문입니다.

이번 연습에서는 위키피디아 텍스트에서 키워드와 단어 구름 형태로 의미를 추출하는 간단한 과정을 거쳤습니다. 이 예시는 매우 간단하지만, 데이터 과학자가 데이터를 다룰 때 데이터 수집부터 시각화까지 거치는 전형적인 모든 단계를 잘 보여줍니다.

우리 강의에서는 이 모든 단계를 자세히 다룰 예정입니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
